# 18,144 episodes: local win rate ranks backwards

I ran two of my own kaggriculture agents against a 15-opponent pool, 2,880 paired episodes
each. `v9` beat `v8` on local self-play by eight points. On the public leaderboard `v9` is
**behind**.

This notebook reproduces that from the per-episode data in
[Kaggriculture: 18,144 Measured Episodes](https://www.kaggle.com/datasets/dariushafshar/kaggriculture-agent-benchmark-18k),
along with the two configuration levers that *do* survive a correct paired test, and the
statistical mistake that produced a false positive before I caught it.

Fork it and rerun the tests on your own arms — every number below is computed from
`episodes.csv` in this cell block, not read from a summary.

In [ ]:
import os, sys, math, collections, csv

# Kaggle's input mount layout is not stable: walk (bounded) for the marker file.
MARKER = "episodes.csv"
root = None
for base in ("/kaggle/input", "input", "."):
    if not os.path.isdir(base):
        continue
    for dirpath, dirnames, filenames in os.walk(base):
        if dirpath.count(os.sep) - base.count(os.sep) > 3:
            dirnames[:] = []
            continue
        if MARKER in filenames:
            root = dirpath
            break
    if root:
        break

if root is None:
    sys.exit(f"FATAL: could not find {MARKER} under /kaggle/input. "
             "Attach dariushafshar/kaggriculture-agent-benchmark-18k and rerun.")
print("dataset root:", root)

rows = []
with open(os.path.join(root, "episodes.csv")) as fh:
    for r in csv.DictReader(fh):
        r["seed"] = int(r["seed"]); r["swap"] = int(r["swap"])
        r["ours"] = float(r["ours"]); r["theirs"] = float(r["theirs"])
        r["win"] = int(r["win"])
        rows.append(r)

print(f"{len(rows):,} episodes loaded")
assert len(rows) == 18144, len(rows)
# the win column is not taken on trust
assert all(r["win"] == int(r["ours"] > r["theirs"]) for r in rows)
print("win column verified against both bank columns for every row")

## 1. The inversion

Local pool win rate versus the public leaderboard score for the same two submissions.

In [ ]:
def wilson(k, n, z=1.96):
    p = k / n; d = 1 + z*z/n
    c = p + z*z/(2*n)
    h = z * math.sqrt(p*(1-p)/n + z*z/(4*n*n))
    return (c-h)/d, (c+h)/d

LADDER = {"v9": 752.7, "v8": 758.6}          # Kaggle submissions API, 2026-08-08 05:57 UTC
pw = [r for r in rows if r["sweep"] == "pool_wide"]

print(f"{'agent':6s} {'local win rate (n=2880)':26s} {'vs opp_c03':>11s} {'ladder':>8s}")
for ag in ("v9", "v8"):
    rs = [r for r in pw if r["label"] == ag]
    k, n = sum(r["win"] for r in rs), len(rs)
    lo, hi = wilson(k, n)
    c03 = [r for r in rs if r["opp"] == "opp_c03"]
    print(f"{ag:6s} {k/n:6.1%} [{lo:.1%}, {hi:.1%}]{'':6s} "
          f"{sum(r['win'] for r in c03)/len(c03):10.1%} {LADDER[ag]:8.1f}")

print("\nLocal ranks v9 first. The ladder ranks v8 first.")

## 2. Why: the pool is doing the talking

Nine of fifteen opponents are at exactly 0.0% and three at exactly 100.0%. An "overall win
rate" over this pool is mostly a readout of pool composition.

In [ ]:
print(f"{'opponent':16s} {'v9':>8s} {'v8':>8s}")
seen = []
for opp in sorted({r["opp"] for r in pw}):
    cell = {}
    for ag in ("v9", "v8"):
        rs = [r for r in pw if r["opp"] == opp and r["label"] == ag]
        cell[ag] = sum(r["win"] for r in rs) / len(rs)
    seen.append((opp, cell))
for opp, cell in sorted(seen, key=lambda t: -t[1]["v9"]):
    print(f"{opp:16s} {cell['v9']:8.1%} {cell['v8']:8.1%}")

rates = [c["v9"] for _, c in seen]
print(f"\nat 0.0%: {sum(r == 0 for r in rates)} of {len(rates)}   "
      f"at 100.0%: {sum(r == 1 for r in rates)} of {len(rates)}")

## 3. The test that matters: McNemar, not two-proportion

Both arms of every comparison here ran the **identical seed set in both seatings**, so they
are paired and the correct test is McNemar on the discordant pairs. The cell below runs the
right test and the wrong one side by side.

The two disagree a lot, and — this is the part worth internalising — **not always in the
same direction.** For `wheat_late_tiles=10` the unpaired test badly understates the effect
(+1.49 vs +2.89) because almost every pair is concordant and the unpaired test cannot see
that. For an earlier farm-hand sweep in this campaign (a different seed set, not included
in this dataset) the unpaired test *overstated* it: z = 2.69 unpaired, and the effect did
not survive pairing.

So the rule is not "unpaired is conservative" or "pairing is more powerful". Neither is
true in general. The rule is: **run the test your design actually implies, and if you
change designs, recompute.** McNemar discards concordant pairs, which can cost you most of
your sample — see the tie counts below.

In [ ]:
def mcnemar(cand, base, key=("opp", "seed", "swap")):
    idx = {tuple(r[k] for k in key): r["win"] for r in base}
    W = L = tie = 0
    for r in cand:
        b = idx.get(tuple(r[k] for k in key))
        if b is None: continue
        if r["win"] == b: tie += 1
        elif r["win"] == 1: W += 1
        else: L += 1
    z = (W - L) / math.sqrt(W + L) if W + L else float("nan")
    return W, L, tie, z

def two_prop(cand, base):
    """The WRONG test for these arms. Shown so you can see the size of the error."""
    k1, n1 = sum(r["win"] for r in cand), len(cand)
    k2, n2 = sum(r["win"] for r in base), len(base)
    p = (k1 + k2) / (n1 + n2)
    se = math.sqrt(p * (1 - p) * (1/n1 + 1/n2))
    return (k1/n1 - k2/n2) / se if se else float("nan")

def arm(sweep, label, opp=None):
    return [r for r in rows if r["sweep"] == sweep and r["label"] == label
            and (opp is None or r["opp"] == opp)]

print(f"{'comparison':38s} {'base':>7s} {'cand':>7s} {'W':>4s} {'L':>4s} "
      f"{'ties':>5s} {'McNemar z':>10s} {'unpaired z':>11s}")
for sweep, label, opp in [
    ("mix_recheck",   "max_quadrants=2",     "opp_c03"),
    ("mix_recheck",   "wheat_late_tiles=10", "opp_c03"),
    ("mix_recheck",   "melon_tiles=18",      "opp_c03"),
    ("mix_recheck",   "care_mult=0.5",       "opp_c03"),
    ("hire_frontier", "cap12_marg100",       "opp_c03"),
]:
    base = arm(sweep, "v9-base", opp); cand = arm(sweep, label, opp)
    W, L, tie, z = mcnemar(cand, base)
    b = sum(r["win"] for r in base)/len(base); c = sum(r["win"] for r in cand)/len(cand)
    print(f"{label + ' vs ' + opp:38s} {b:7.1%} {c:7.1%} {W:4d} {L:4d} {tie:5d} "
          f"{z:+10.2f} {two_prop(cand, base):+11.2f}")

print("\nmax_quadrants=2 is the real lever: +23.9 points, z=+4.00, 28 W / 5 L.")
print("cap12_marg100 clears NEITHER test (z=+1.15 paired) -- 67 of 128 pairs are ties,")
print("so most of the sample carries no information about the comparison at all.")

## 4. Shadowed parameters — the knob that could not have done anything

Some swept values returned banks **byte-identical** to baseline on every single episode.
That is a different claim from "no effect": the parameter was structurally unreachable.

Hand count is `min(max_hands, largest k with fib(k) <= hire_marginal_max)`. Wages are
Fibonacci and `fib(11) = 89`, so both gates have to open together — moving either one alone
does literally nothing. Check your own agent for this shape before you spend a night
sweeping a knob that is shadowed.

In [ ]:
for sweep in ("mix_recheck", "hire_frontier"):
    base = {(r["opp"], r["seed"], r["swap"]): r["ours"]
            for r in rows if r["sweep"] == sweep and r["label"] == "v9-base"}
    print(f"\n[{sweep}]")
    for label in sorted({r["label"] for r in rows if r["sweep"] == sweep}):
        if label == "v9-base": continue
        cand = arm(sweep, label)
        ident = sum(1 for r in cand if base.get((r["opp"], r["seed"], r["swap"])) == r["ours"])
        if ident == len(cand):
            print(f"  SHADOWED  {label:24s} bank identical on all {len(cand)} episodes")

## What to take from this

1. **Screen with a pool you have reason to believe is representative**, and report the
   per-opponent breakdown, not just the aggregate. Mine had 12 of 15 opponents contributing
   no information at all.
2. **If your arms share seeds, use McNemar** — and check the tie count. The unpaired test
   disagrees materially here, in both directions depending on the comparison.
3. **Check whether a parameter is reachable before you sweep it.** Byte-identical banks are
   the cheapest possible diagnostic and they cost one comparison.
4. **A local win-rate improvement is a hypothesis, not a result.** Mine ranked two agents
   backwards against the actual leaderboard.

Data: [Kaggriculture: 18,144 Measured Episodes](https://www.kaggle.com/datasets/dariushafshar/kaggriculture-agent-benchmark-18k) ·
engine constants: [Kaggriculture Engine Reference Tables](https://www.kaggle.com/datasets/dariushafshar/kaggriculture-engine-reference-tables)